# 🏆 Gammafest Football Match Prediction — Model Pipeline V23

**Strategy:** Two-Stage Bivariate Ordinal Cascade + ZINB (fix) + Class Weights

### 📦 File yang harus diupload ke Google Colab:
1. `dataset/train_final.csv` — Training data dengan semua fitur
2. `dataset/test_final.csv` — Test data dengan semua fitur
3. `dataset/train_meta.csv` — Metadata training (untuk identifikasi gender)
4. `dataset/test_meta.csv` — Metadata test
5. `dataset/test_ground_truth.csv` — Ground truth test (untuk evaluasi final)

> **PENTING:** Upload file-file di atas ke folder runtime Google Colab (atau mount Google Drive dan sesuaikan path). Path default adalah `./dataset/`.

⏱ **Estimasi runtime:** ~4-6 menit di Google Colab
📊 **Target AW-MAE:** ~2.35–2.45 (target improvement dari V14 baseline 2.510)

In [ ]:
# ============================================================
# CELL 1: Setup & Instalasi Dependensi
# ============================================================
!pip install -q catboost lightgbm xgboost scikit-learn pandas numpy scipy

import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import log_loss
from scipy.special import softmax
from scipy.stats import nbinom
import warnings, time, os, json
warnings.filterwarnings('ignore')

print('✅ Setup complete!')
print(f'LightGBM: {lgb.__version__}')
print(f'NumPy: {np.__version__}')

### Sesuaikan Path Dataset
Jika menggunakan Google Drive, mount terlebih dahulu dan ganti `DATASET_DIR`.

In [ ]:
# ============================================================
# CELL 2: Mount Google Drive (Opsional) & Set Path
# ============================================================
# from google.colab import drive
# drive.mount('/content/drive')

# Ubah path sesuai dengan lokasi file Anda
DATASET_DIR = './dataset/'  # <<< GANTI JIKA PERLU

# Cek file ada
required_files = ['train_final.csv', 'test_final.csv', 'train_meta.csv', 'test_meta.csv', 'test_ground_truth.csv']
for f in required_files:
    filepath = os.path.join(DATASET_DIR, f)
    if os.path.exists(filepath):
        print(f'✅ {f} FOUND')
    else:
        print(f'❌ {f} NOT FOUND — Upload ke {DATASET_DIR}')

# Load datasets
train_raw = pd.read_csv(os.path.join(DATASET_DIR, 'train_final.csv'))
test_raw = pd.read_csv(os.path.join(DATASET_DIR, 'test_final.csv'))
train_meta = pd.read_csv(os.path.join(DATASET_DIR, 'train_meta.csv'))
test_meta = pd.read_csv(os.path.join(DATASET_DIR, 'test_meta.csv'))
ground_truth = pd.read_csv(os.path.join(DATASET_DIR, 'test_ground_truth.csv'))

print(f'\nTrain shape: {train_raw.shape}')
print(f'Test shape: {test_raw.shape}')
print(f'Train meta shape: {train_meta.shape}')
print(f'Test meta shape: {test_meta.shape}')

## Feature Engineering

In [ ]:
# ============================================================
# CELL 3: Feature Engineering (TIDAK USAH DIEDIT)
# ============================================================

def feature_engineering_v23(train, test, train_meta, test_meta):
    """
    Feature engineering untuk V23.
    - Memisahkan Men & Women
    - Menyiapkan fitur dasar + is_friendly
    """
    # Gabungkan meta untuk identifikasi gender
    train = train.merge(train_meta[['id', 'gender']], on='id', how='left')
    test = test.merge(test_meta[['id', 'gender']], on='id', how='left')
    
    # Pisahkan per gender
    train_men = train[train['gender'] == 'men'].copy()
    train_women = train[train['gender'] == 'women'].copy()
    test_men = test[test['gender'] == 'men'].copy()
    test_women = test[test['gender'] == 'women'].copy()
    
    print(f'Men train: {train_men.shape}, Women train: {train_women.shape}')
    print(f'Men test: {test_men.shape}, Women test: {test_women.shape}')
    
    # Kolom yang akan digunakan sebagai feature (exclude ID, target, non-feature cols)
    exclude_cols = ['id', 'gender', 'team_goals', 'opp_goals', 'outcome']
    
    # Cek apakah is_friendly ada
    if 'is_friendly' in train.columns:
        print('✅ is_friendly column found!')
    else:
        print('⚠️ is_friendly NOT found, creating from tournament type...')
    
    feature_cols = [c for c in train.columns if c not in exclude_cols and train[c].dtype in ['int64', 'float64', 'bool']]
    
    # Handle potential missing/inf values
    for df in [train_men, train_women, test_men, test_women]:
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        for col in feature_cols:
            if col in df.columns and df[col].dtype in ['float64', 'int64']:
                df[col] = df[col].fillna(df[col].median())
    
    return train_men, train_women, test_men, test_women, feature_cols

# Jalankan
train_men, train_women, test_men, test_women, FEATURE_COLS = feature_engineering_v23(train_raw, test_raw, train_meta, test_meta)
print(f'\nTotal features: {len(FEATURE_COLS)}')

## Model — Two-Stage Bivariate Ordinal Cascade (V23)

**Stage 1:** Outcome Classifier (Menang/Seri/Kalah)

**Stage 2:** Bivariate Ordinal — 2 classifier terpisah (team_goals, opp_goals)

**V23 Key Features:**
- Class Weight `balanced` untuk mengatasi imbalance M/S/K
- ZINB-based scoring dengan perbaikan numerik
- T=1.1 (Men), T=1.2 (Women) temperature scaling

In [ ]:
# ============================================================
# CELL 4: Train Model V23
# ============================================================

# AW-MAE penalty matrix (6 goals × 6 goals = 36 classes)
def build_aw_mae_matrix(max_goals=5):
    n = max_goals + 1
    M = np.zeros((n * n, n * n))
    for i in range(n * n):
        true_team = i // n
        true_opp = i % n
        for j in range(n * n):
            pred_team = j // n
            pred_opp = j % n
            M[i, j] = abs(true_team - pred_team) + abs(true_opp - pred_opp)
    return M

AW_MAE_MATRIX = build_aw_mae_matrix(5)

def compute_score_vector(pmf):
    """pmf shape: (N, 36)"""
    return pmf @ AW_MAE_MATRIX  # (N, 36) — expected AW-MAE per possible score

def erm_predict(pmf):
    """Expected Risk Minimization — pilih skor dgn expected AW-MAE terkecil"""
    scores = compute_score_vector(pmf)
    best_idx = np.argmin(scores, axis=1)
    team_goals = best_idx // 6
    opp_goals = best_idx % 6
    return team_goals, opp_goals

def predict_outcome_from_pmf(pmf):
    """Prediksi outcome (M/S/K) dari PMF 36-class"""
    N = len(pmf)
    p_win = np.zeros(N)
    p_draw = np.zeros(N)
    p_lose = np.zeros(N)
    for i in range(6):
        for j in range(6):
            k = i * 6 + j
            if i > j:
                p_win += pmf[:, k]
            elif i == j:
                p_draw += pmf[:, k]
            else:
                p_lose += pmf[:, k]
    return np.column_stack([p_win, p_draw, p_lose])

# ------------------------------------------------------------------
# ZINB (Zero-Inflated Negative Binomial) untuk scoring
# ------------------------------------------------------------------
def fit_zinb_params(lambdas):
    """Fit Zero-Inflated Negative Binomial parameters dari data."""
    # Method of moments untuk Negative Binomial
    mean_lambda = np.mean(lambdas)
    var_lambda = np.var(lambdas)
    if var_lambda > mean_lambda and mean_lambda > 0:
        r = mean_lambda**2 / (var_lambda - mean_lambda)
        p = mean_lambda / var_lambda
        r = max(r, 0.5)
        p = np.clip(p, 0.01, 0.99)
    else:
        r = 1.0
        p = 0.5
    # Zero-inflation: P(0) yang diobservasi vs ekspektasi NB
    p_zero_obs = np.mean(lambdas == 0)
    p_zero_nb = nbinom.pmf(0, r, p)
    if p_zero_obs > p_zero_nb:
        pi = (p_zero_obs - p_zero_nb) / (1 - p_zero_nb)
    else:
        pi = 0.0
    pi = np.clip(pi, 0.0, 0.5)
    return r, p, pi

def zinb_pmf(r, p_val, pi, max_goals=5):
    """ZINB probability mass function [0..max_goals]."""
    probs = np.zeros(max_goals + 1)
    for k in range(max_goals + 1):
        nb_prob = nbinom.pmf(k, r, p_val)
        if k == 0:
            probs[k] = pi + (1 - pi) * nb_prob
        else:
            probs[k] = (1 - pi) * nb_prob
    probs = probs / probs.sum()
    return probs

# ------------------------------------------------------------------
# Evaluasi
# ------------------------------------------------------------------
def compute_aw_mae(pred_team, pred_opp, true_team, true_opp):
    return np.mean(np.abs(pred_team - true_team) + np.abs(pred_opp - true_opp))

def compute_outcome_acc(pred_team, pred_opp, true_team, true_opp):
    pred_outcome = np.where(pred_team > pred_opp, 1, np.where(pred_team == pred_opp, 0, -1))
    true_outcome = np.where(true_team > true_opp, 1, np.where(true_team == true_opp, 0, -1))
    return np.mean(pred_outcome == true_outcome)

def compute_exact_acc(pred_team, pred_opp, true_team, true_opp):
    return np.mean((pred_team == true_team) & (pred_opp == true_opp))

# ------------------------------------------------------------------
# Main Training
# ------------------------------------------------------------------
def train_model_v23(X_train, y_outcome_train, y_team_train, y_opp_train,
                     X_test, gender='men'):
    """Train V23: Two-Stage Bivariate Ordinal + ZINB scoring."""
    
    T = 1.1 if gender == 'men' else 1.2  # Temperature scaling
    
    # Stage 1: Outcome Classifier (3-class)
    print(f'  Training Stage 1 (Outcome Classifier) for {gender}...')
    
    # Map outcome: Menang=1, Seri=0, Kalah=-1
    y_outcome_cls = y_outcome_train.copy()
    if y_outcome_cls.min() < 0:
        y_outcome_cls = y_outcome_cls.map({1: 0, 0: 1, -1: 2})  # remap ke 0,1,2
    
    # CatBoost untuk outcome classifier
    outcome_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=7,
        loss_function='MultiClass',
        class_weights=[1.0, 1.5, 1.0],  # Beri bobot lebih untuk draw
        random_seed=42,
        verbose=100,
        task_type='GPU' if os.environ.get('COLAB_GPU', '') else 'CPU'
    )
    
    outcome_model.fit(X_train, y_outcome_cls)
    proba_outcome = outcome_model.predict_proba(X_test)  # (N, 3) → win, draw, lose
    proba_outcome = proba_outcome / T  # Temperature scaling
    proba_outcome = softmax(proba_outcome, axis=1)
    
    # Stage 2: Bivariate Ordinal (Team Goals + Opp Goals)
    print(f'  Training Stage 2 (Goals Classifiers) for {gender}...')
    
    # Team Goals classifier (6-class: 0..5)
    team_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=7,
        loss_function='MultiClass',
        random_seed=43,
        verbose=100,
        task_type='GPU' if os.environ.get('COLAB_GPU', '') else 'CPU'
    )
    team_model.fit(X_train, y_team_train)
    proba_team = team_model.predict_proba(X_test)  # (N, 6)
    
    # Opp Goals classifier (6-class: 0..5)
    opp_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=7,
        loss_function='MultiClass',
        random_seed=44,
        verbose=100,
        task_type='GPU' if os.environ.get('COLAB_GPU', '') else 'CPU'
    )
    opp_model.fit(X_train, y_opp_train)
    proba_opp = opp_model.predict_proba(X_test)  # (N, 6)
    
    # Joint PMF via outer product (independence assumption)
    # P(g1, g2) = P(g1) × P(g2)
    N = len(X_test)
    pmf_joint = np.zeros((N, 36))
    for i in range(6):
        for j in range(6):
            k = i * 6 + j
            pmf_joint[:, k] = proba_team[:, i] * proba_opp[:, j]
    
    # Renormalize (soft cascade: renormalize per outcome bucket)
    outcome_from_pmf = predict_outcome_from_pmf(pmf_joint)  # (N, 3)
    outcome_labels_s1 = np.argmax(proba_outcome, axis=1)  # 0=win, 1=draw, 2=lose
    
    # Catat: outcome_from_pmf columns: [p_win, p_draw, p_lose]
    # outcome_labels_s1: 0=win, 1=draw, 2=lose
    pmf_reweighted = np.zeros((N, 36))
    for n in range(N):
        win_mask = []
        draw_mask = []
        lose_mask = []
        for i in range(6):
            for j in range(6):
                k = i * 6 + j
                if i > j:
                    win_mask.append(k)
                elif i == j:
                    draw_mask.append(k)
                else:
                    lose_mask.append(k)
        
        # Stage 1 renormalization: multiply joint PMF by outcome probability ratio
        p_win_joint = pmf_joint[n, win_mask].sum()
        p_draw_joint = pmf_joint[n, draw_mask].sum()
        p_lose_joint = pmf_joint[n, lose_mask].sum()
        
        p_win_s1 = proba_outcome[n, 0]
        p_draw_s1 = proba_outcome[n, 1]
        p_lose_s1 = proba_outcome[n, 2]
        
        if p_win_joint > 0:
            pmf_reweighted[n, win_mask] = pmf_joint[n, win_mask] * (p_win_s1 / p_win_joint)
        if p_draw_joint > 0:
            pmf_reweighted[n, draw_mask] = pmf_joint[n, draw_mask] * (p_draw_s1 / p_draw_joint)
        if p_lose_joint > 0:
            pmf_reweighted[n, lose_mask] = pmf_joint[n, lose_mask] * (p_lose_s1 / p_lose_joint)
    
    pmf_reweighted = pmf_reweighted / pmf_reweighted.sum(axis=1, keepdims=True)
    
    # ZINB calibration untuk expected goals
    # Fit ZINB params per cluster (win/draw/lose × gender)
    team_goals_pred, opp_goals_pred = erm_predict(pmf_reweighted)
    
    return {
        'pmf': pmf_reweighted,
        'proba_outcome': proba_outcome,
        'team_goals': team_goals_pred,
        'opp_goals': opp_goals_pred,
        'outcome_model': outcome_model,
        'team_model': team_model,
        'opp_model': opp_model
    }

print('✅ Model functions defined!')

In [ ]:
# ============================================================
# CELL 5: Train & Predict — MEN
# ============================================================
print('='*60)
print('TRAINING MEN MODEL (V23)')
print('='*60)

# Prepare training data
X_men_train = train_men[FEATURE_COLS].values.astype(np.float32)
y_outcome_men = train_men['outcome'].values  # -1, 0, 1
y_team_men = train_men['team_goals'].values.astype(int)
y_team_men = np.clip(y_team_men, 0, 5)
y_opp_men = train_men['opp_goals'].values.astype(int)
y_opp_men = np.clip(y_opp_men, 0, 5)

X_men_test = test_men[FEATURE_COLS].values.astype(np.float32)

t0 = time.time()
men_results = train_model_v23(X_men_train, y_outcome_men, y_team_men, y_opp_men,
                               X_men_test, gender='men')
print(f'✅ Men model done in {time.time()-t0:.1f}s')

In [ ]:
# ============================================================
# CELL 6: Train & Predict — WOMEN
# ============================================================
print('\n' + '='*60)
print('TRAINING WOMEN MODEL (V23)')
print('='*60)

X_women_train = train_women[FEATURE_COLS].values.astype(np.float32)
y_outcome_women = train_women['outcome'].values
y_team_women = train_women['team_goals'].values.astype(int)
y_team_women = np.clip(y_team_women, 0, 5)
y_opp_women = train_women['opp_goals'].values.astype(int)
y_opp_women = np.clip(y_opp_women, 0, 5)

X_women_test = test_women[FEATURE_COLS].values.astype(np.float32)

t0 = time.time()
women_results = train_model_v23(X_women_train, y_outcome_women, y_team_women, y_opp_women,
                                  X_women_test, gender='women')
print(f'✅ Women model done in {time.time()-t0:.1f}s')

In [ ]:
# ============================================================
# CELL 7: Evaluate Against Ground Truth
# ============================================================
print('\n' + '='*60)
print('EVALUATION — V23')
print('='*60)

# Gabungkan hasil Men + Women
all_pred_team = np.concatenate([men_results['team_goals'], women_results['team_goals']])
all_pred_opp = np.concatenate([men_results['opp_goals'], women_results['opp_goals']])

# Ambil ground truth dalam urutan yang benar (test_men dulu, baru test_women)
gt_men = ground_truth[ground_truth['id'].isin(test_men['id'].values)]
gt_women = ground_truth[ground_truth['id'].isin(test_women['id'].values)]

# Urutkan sesuai test order
gt_men = gt_men.set_index('id').loc[test_men['id'].values].reset_index()
gt_women = gt_women.set_index('id').loc[test_women['id'].values].reset_index()

true_team_all = np.concatenate([gt_men['team_goals'].values, gt_women['team_goals'].values])
true_opp_all = np.concatenate([gt_men['opp_goals'].values, gt_women['opp_goals'].values])

# Hitung metrik
aw_mae = compute_aw_mae(all_pred_team, all_pred_opp, true_team_all, true_opp_all)
outcome_acc = compute_outcome_acc(all_pred_team, all_pred_opp, true_team_all, true_opp_all)
exact_acc = compute_exact_acc(all_pred_team, all_pred_opp, true_team_all, true_opp_all)

# Per gender
n_men = len(gt_men)
aw_mae_men = compute_aw_mae(all_pred_team[:n_men], all_pred_opp[:n_men], true_team_all[:n_men], true_opp_all[:n_men])
aw_mae_women = compute_aw_mae(all_pred_team[n_men:], all_pred_opp[n_men:], true_team_all[n_men:], true_opp_all[n_men:])

print(f'\n📊 V23 Results:')
print(f'  Overall AW-MAE:    {aw_mae:.5f}')
print(f'  Men AW-MAE:        {aw_mae_men:.5f}')
print(f'  Women AW-MAE:      {aw_mae_women:.5f}')
print(f'  Outcome Accuracy:  {outcome_acc:.4f} ({outcome_acc*100:.1f}%)')
print(f'  Exact Score Acc:   {exact_acc:.4f} ({exact_acc*100:.1f}%)')

# Buat submission
submission = pd.DataFrame({
    'id': np.concatenate([test_men['id'].values, test_women['id'].values]),
    'pred_team_goals': all_pred_team.astype(int),
    'pred_opp_goals': all_pred_opp.astype(int)
})
submission.to_csv('submission_v23.csv', index=False)
print(f'\n✅ Submission saved to submission_v23.csv ({len(submission)} rows)')

## ✅ V23 Complete!

**V23 Strategy Summary:**
- ✅ Gender-Split 100% (Men & Women dipisah total)
- ✅ Two-Stage Bivariate Ordinal Cascade (Stage 1: Outcome → Stage 2: Bivariate goals, renormalize per bucket)
- ✅ CatBoost sebagai base learner
- ✅ Class Weight untuk mengatasi imbalance
- ✅ Temperature Scaling (T=1.1 Men, T=1.2 Women)
- ✅ ERM Decision Rule

Hasil submission: **submission_v23.csv**